# PACE MiniGrid pilot on one Kaggle GPU

Import this notebook, select a **GPU** accelerator, enable **Internet**, and run all cells. The notebook clones the pinned source and generates MiniGrid levels during training.

The pilot uses the paper preset (15×15 Maze, 32 environments, 256 steps per rollout) for **24 main-loop rollout batches** after prefilling the buffer with 500 scored levels. It saves checkpoints every 8 batches, exits at batch 16, and starts a fresh process to resume to batch 24. The output directory is under `/kaggle/working/pace/`.

The pilot uses a provisional step size of 0.01 as a documented test choice. Table 9 does not specify the provisional step size.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
REPO_URL = 'https://github.com/SimforianusJonathan/PACE-Research.git'
REPO_REF = '178a8e85813ea94f96077aaece05f3ebf0987b8f'
REPO = WORK / f"PACE-Research-{REPO_REF.replace('/', '-')[:12]}"
SOURCE = ''  # Optional path to an uploaded source directory or Git archive ZIP.
RUNTIME = Path('/tmp/pace-kaggle')

if not REPO.exists():
    if SOURCE:
        source = Path(SOURCE)
        if source.is_dir():
            shutil.copytree(source, REPO)
        else:
            with zipfile.ZipFile(source) as archive:
                archive.extractall(REPO)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
        subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO, check=True)

if not (REPO / 'scripts/setup_kaggle.sh').is_file():
    raise RuntimeError('This source lacks Kaggle support. Use a Git ref or source archive containing this notebook.')
if (REPO / '.git').exists():
    subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True)

# Use Kaggle's unmodified driver environment for this diagnostic.
smi = shutil.which('nvidia-smi')
if not smi or subprocess.run([smi, '-L'], check=False).returncode != 0:
    print('nvidia-smi did not list a GPU. Checking the CUDA driver directly.')

visible_device = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',', 1)[0]
env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': visible_device,
       'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
       'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
# Keep mounted driver libraries while letting the wheels provide the CUDA toolkit.
paths = ['/usr/local/nvidia/lib64', *os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep)]
driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                            and Path(path).name not in {'compat', 'stubs'}
                            and (Path(path) / 'libcuda.so.1').is_file())
if driver_paths:
    env['LD_LIBRARY_PATH'] = os.pathsep.join(driver_paths)
else:
    env.pop('LD_LIBRARY_PATH', None)

driver_probe = '''
import ctypes
cuda = ctypes.CDLL('libcuda.so.1')
status = cuda.cuInit(0)
if status != 0:
    raise RuntimeError(f'CUDA driver initialization failed (cuInit={status})')
count = ctypes.c_int()
status = cuda.cuDeviceGetCount(ctypes.byref(count))
if status != 0 or count.value < 1:
    raise RuntimeError(f'No visible CUDA device (status={status}, count={count.value})')
'''
try:
    subprocess.run([sys.executable, '-c', driver_probe], env=env, check=True)
except subprocess.CalledProcessError:
    raise RuntimeError('Kaggle has no usable CUDA device for this run. Select a GPU accelerator, '
                       'then restart the session or save a new GPU-enabled version.') from None

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace',
    ) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f'{log_path.stem} failed with exit code {code}. Full log: {log_path}')

run_logged(['bash', 'scripts/setup_kaggle.sh'], WORK / 'pace' / 'setup.log')
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
from jaxued.environments import Maze
from train.pace_minigrid import MiniGridConfig
if getattr(MiniGridConfig.smoke(), 'protocol', None) != 'algorithm1':
    raise RuntimeError('Select a repository revision with the algorithm1 training protocol.')
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin, 'environment': Maze.__name__})
'''
run_logged([PACE_PYTHON, '-c', probe], WORK / 'pace' / 'gpu-probe.log')

In [ ]:
PILOT_SEED = 0
PROVISIONAL_ALPHA = 0.01  # Explicit pilot choice; the paper leaves this unspecified.
CHECKPOINT_INTERVAL = 8
PAUSE_AFTER_BATCH = 16
FINISH_AFTER_BATCH = 24

runs = WORK / 'pace'
runs.mkdir(exist_ok=True)
run = Path(tempfile.mkdtemp(prefix='pilot-', dir=runs))
base = [PACE_PYTHON, '-m', 'train.pace_minigrid']
check_args = ['--device', 'gpu', '--output-dir', str(run),
              '--checkpoint-interval', str(CHECKPOINT_INTERVAL)]
run_logged(base + ['train', '--seed', str(PILOT_SEED), '--budget', 'ppo', '--protocol', 'algorithm1',
                   '--provisional-alpha', str(PROVISIONAL_ALPHA),
                   '--stop-after', str(PAUSE_AFTER_BATCH)] + check_args, run / 'train.log')

boundary = run / 'checkpoints' / f'step_{PAUSE_AFTER_BATCH:08d}.msgpack'
if not boundary.is_file():
    raise RuntimeError(f'Expected pilot checkpoint: {boundary}')
print('Pilot checkpoint:', boundary)

In [ ]:
run_logged(base + ['resume', '--checkpoint', str(boundary),
                   '--stop-after', str(FINISH_AFTER_BATCH)] + check_args, run / 'resume.log')

final = run / 'checkpoints' / f'step_{FINISH_AFTER_BATCH:08d}.msgpack'
if not final.is_file():
    raise RuntimeError(f'Expected resumed checkpoint: {final}')
print('Resumed pilot checkpoint:', final)

In [ ]:
verification = '''
import json
import sys
from pathlib import Path

from train.pace_checkpoint import read_checkpoint
from train.pace_minigrid import MiniGridConfig, interaction_counts

path = Path(sys.argv[1])
expected_batches = int(sys.argv[2])
payload = read_checkpoint(path)
config = MiniGridConfig(**payload['config'])
runner = payload['runner']
student = runner['student']
rollouts = int(runner['rollout_batches'])
ppo = int(runner['ppo_batches'])
optimizer_steps = int(student['step'])
occupancy = int(student['sampler']['size'])
expected_optimizer_steps = ppo * config.ppo_epochs * config.num_minibatches

if rollouts != expected_batches or not 0 < ppo <= rollouts or optimizer_steps != expected_optimizer_steps:
    raise RuntimeError('Pilot counters are inconsistent or no replay PPO batch ran.')
if (config.protocol, config.budget, config.maze_size, config.num_envs, config.num_steps) != ('algorithm1', 'ppo', 15, 32, 256):
    raise RuntimeError('Pilot checkpoint does not use the paper rollout preset.')
if config.initial_buffer_size != 500 or occupancy < 500 or int(runner['prefill_env_steps']) < 500 * config.num_steps:
    raise RuntimeError('Pilot checkpoint does not account for the scored 500-level prefill.')

print(json.dumps({
    'checkpoint': str(path),
    'rollout_batches': rollouts,
    'ppo_batches': ppo,
    'optimizer_steps': optimizer_steps,
    **interaction_counts(config, rollouts, ppo, runner['prefill_env_steps']),
    'buffer_size': occupancy,
    'provisional_alpha': config.provisional_alpha,
}, indent=2))
'''
run_logged([PACE_PYTHON, '-c', verification, str(final), str(FINISH_AFTER_BATCH)], run / 'verify.log')